In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import asyncio

In [9]:
load_dotenv(override=True)

True

In [10]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [11]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-5-nano")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-5-nano")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-5-nano")

In [12]:
result = Runner.run_streamed(sales_agent1, input="write a cold sales email")  # we're not using await here but in below line is async for
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Subject: Simplify SOC 2 readiness for {Company} with AI

Hi {FirstName},

I’m {YourName} from ComplAI. We help fast-growing teams prepare for SOC 2 audits with an AI-powered platform that:
- automates evidence collection and control mapping
- centralizes audit artifacts
- provides continuous compliance monitoring

This can shorten audit cycles and reduce last-minute auditor questions. I’d love to show you a 15-minute demo and discuss how it fits {Company}’s controls and timeline.

Are you available {Date/Time} or {Alternative}?

Best regards,
{YourName}
{Title}
ComplAI
{Phone}
{Email}
{Website}

P.S. If you’re not the right person, could you connect me with the SOC 2 owner?

In [13]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-5-nano"
)

In [ ]:
message = "Write a cold sales email"
with trace("selecting the best email"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "cold emails: \n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)

    print(f"best email: {best.final_output}")


In [15]:
# using tools:
@function_tool
def send_email(body: str):
    """send out an email"""
    print("==body==", body)
    print("email sent successfully")
    return {"status": "success"}
    

In [16]:
# above decorator @function_tool will turn a function into a tool, but you can also convert an agent into a tool:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11542e270>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [17]:
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description="write a cold sales email")
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description="write a cold sales email")

tools = [tool1, tool2, tool3, send_email]


In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

agent = Agent(name="Sales Manager", tools=tools, instructions=instructions, model="gpt-5-nano")
message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(agent, message)
    print("==result", result.final_output)

In [20]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-5-nano")
subject_writer_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [21]:
@function_tool
def send_email(subject: str, html_body: str):
    """send out an email"""
    print("==subject and body==", subject, "\n" , html_body)
    print("email sent successfully")
    return {"status": "success"}

In [22]:
tools = [subject_writer_tool, html_tool]

In [23]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(name="emailer agent", instructions=instructions, model="gpt-5-nano", tools=tools, handoff_description="Convert an email to html and send it")
# The handoff_description is metadata that tells the orchestrator: "This agent is responsible for tasks involving converting emails to HTML and sending them."



In [24]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]



In [25]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'emailer agent' agent. The 'emailer agent' will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the 'emailer agent' — never more than one.
"""
sales_manager = Agent(name="sales manager", instructions=sales_manager_instructions, tools=tools, handoffs=handoffs, model="gpt-5-nano")
message = "Send out a cold sales email addressed to Dear CEO from Alice"
with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)
